In [1]:
import os
import pandas as pd 
import re

In [2]:
# Config
DATASET = 'Zebrafish-2024' 
RAW_DIR = os.path.join(DATASET, 'raw-data')
PROCESSED_DIR = os.path.join(DATASET, 'processed-data')

**Make Counts Matrix**

In [ ]:
counts_path = os.path.join(RAW_DIR, "counts_deseq2.txt")
counts_df = pd.read_csv(counts_path, sep="\t", index_col=0)
counts_df.shape
# Counts are currently in genes x samples format for DESeq2, but we need samples x genes for pydeseq2
counts_df = counts_df.T
counts_df.shape
# Now counts are samples x genes (30 samples x 32520 genes)
counts_df.head()
# Change column name (gene_id) to row name (sample)
counts_df.columns.name = None
counts_df.index.name = "sample"
# Save the processed counts to a new file
counts_df = counts_df.reset_index()
processed_counts_path = os.path.join(PROCESSED_DIR, "counts.csv")
counts_df.to_csv(processed_counts_path, index=False)

In [ ]:
newcounts_df = pd.read_csv(processed_counts_path, index_col='sample')
newcounts_df.shape

(30, 32520)

**Make Metadata**

In [ ]:
# Extract sample IDs from the index of the counts dataframe
counts_df = pd.read_csv(processed_counts_path, index_col='sample')
sample_ids = counts_df.index.tolist()

pattern = r"^(.*)_rep(\d+)$"

study_map = {
    "CTRL":     {"drug": "none",      "treatment": "healthy_control"},
    "BMAA":     {"drug": "BMAA",      "treatment": "pathogenic_control"},
    "BMAAals":  {"drug": "CNR401",    "treatment": "treated"},
    "BMAApure": {"drug": "CFA",       "treatment": "treated"},
    "BMAAeda":  {"drug": "Edaravone", "treatment": "treated"}
}

collection_dates = {
    'BMAA_rep1': '2024-05-08', 'BMAA_rep2': '2024-05-07', 'BMAA_rep3': '2024-05-08', 
    'BMAA_rep4': '2024-05-09', 'BMAA_rep5': '2024-05-07', 'BMAA_rep6': '2024-05-07', 
    'BMAAals_rep1': '2024-05-08', 'BMAAals_rep2': '2024-05-08', 'BMAAals_rep3': '2024-05-09', 
    'BMAAals_rep4': '2024-05-07', 'BMAAals_rep5': '2024-05-07', 'BMAAals_rep6': '2024-05-09', 
    'BMAAeda_rep1': '2024-05-09', 'BMAAeda_rep2': '2024-05-07', 'BMAAeda_rep3': '2024-05-08', 
    'BMAAeda_rep4': '2024-05-08', 'BMAAeda_rep5': '2024-05-09', 'BMAAeda_rep6': '2024-05-07', 
    'BMAApure_rep1': '2024-05-09', 'BMAApure_rep2': '2024-05-08', 'BMAApure_rep3': '2024-05-07', 
    'BMAApure_rep4': '2024-05-07', 'BMAApure_rep5': '2024-05-08', 'BMAApure_rep6': '2024-05-09', 
    'CTRL_rep1': '2024-05-09', 'CTRL_rep2': '2024-05-09', 'CTRL_rep3': '2024-05-07', 
    'CTRL_rep4': '2024-05-07', 'CTRL_rep5': '2024-04-17', 'CTRL_rep6': '2024-04-24'
}

metadata_list = []

for sid in counts_df.index:
    match = re.match(pattern, str(sid))
    if match:
        condition_name, rep_num = match.groups()
        # Use the dictionary to look up your specific rules
        # .get() provides a fallback if a name isn't found
        info = study_map.get(condition_name, {"drug": "unknown", "treatment": "unknown"})
        metadata_list.append({
            "sample": sid,
            "drug": info["drug"],
            "treatment": info["treatment"],
            "replicate": int(rep_num),
        })

metadata_df = pd.DataFrame(metadata_list)
metadata_df['date_collected'] = metadata_df['sample'].map(collection_dates)
metadata_df.set_index('sample')

# Batch effects disovered by PCA
batch_map = {
    # Batch 1
    '2024-05-08': 'batch_1',
    # Batch 2
    '2024-05-07': 'batch_2',
    '2024-05-09': 'batch_2',
    # Batch 3
    '2024-04-17': 'batch_3',
    '2024-04-24': 'batch_3'
}
metadata_df['batch'] = metadata_df['date_collected'].map(batch_map)

# Save the metadata to a new file
metadata_path = os.path.join(PROCESSED_DIR, "metadata.csv")
metadata_df.to_csv(metadata_path, index=False)

**Make QC Stats**

In [3]:
# Config 
mapping_file = os.path.join(RAW_DIR, "raw_counts.txt.summary")
counts_file = os.path.join(PROCESSED_DIR, "counts.csv")
qc_out_path = os.path.join(PROCESSED_DIR, "qc_metrics.csv")

'''
mapping_file = "mock-data/mock_summary.txt"
counts_file = "mock-data/mock_counts.csv"
qc_out_path = "mock-data/mock_qc_metrics.csv"
'''

reads_threshold = 1.5e7
mapping_threshold = 0.5
rRNA_threshold = 0.15

In [4]:
# Read the files
mapping_df = pd.read_csv(mapping_file, sep="\t", header=0, index_col=0)
counts_df = pd.read_csv(counts_file, index_col='sample')

# Explore data structure and ensure it is read correctly
mapping_df.head() # Rows are mapping stats and columns are samples
mapping_df.shape # There are 14 mapping stats and 30 samples
mapping_df.columns = counts_df.index # Clean up sample names 

# Extract mapping stats names to a list for future use 
mapping_stats=mapping_df.index.tolist()
print(mapping_stats)

['Assigned', 'Unassigned_Unmapped', 'Unassigned_Read_Type', 'Unassigned_Singleton', 'Unassigned_MappingQuality', 'Unassigned_Chimera', 'Unassigned_FragmentLength', 'Unassigned_Duplicate', 'Unassigned_MultiMapping', 'Unassigned_Secondary', 'Unassigned_NonSplit', 'Unassigned_NoFeatures', 'Unassigned_Overlapping_Length', 'Unassigned_Ambiguity']


In [8]:
qc_df = pd.DataFrame(index=counts_df.index)

# Mapping stats
qc_df['total_reads'] = counts_df.sum(axis=1)
qc_df['total_alignments'] = mapping_df.sum(axis=0) # The above is equivalent
qc_df['mapping_rate'] = mapping_df.loc['Assigned'] / qc_df['total_alignments']
qc_df['n_detected_genes'] = (counts_df > 0).sum(axis=1)
qc_df['multimapping_rate'] = mapping_df.loc['Unassigned_MultiMapping'] / qc_df['total_alignments']

MAD_thresholds = {}
for stat in qc_df.columns:
  median = qc_df[stat].median()
  mad = (qc_df[stat] - median).abs().median() * 1.4826
  lower = median - (2 * mad)
  higher = median + (2 * mad)
  MAD_thresholds[stat] = (lower, higher)
thresholds_df = pd.DataFrame(MAD_thresholds, index=['mad_lower', 'mad_upper']).T

qc_df.head()

,total_reads,total_alignments,mapping_rate,n_detected_genes,multimapping_rate
sample,,,,,
BMAA_rep1,30288241,44204831,0.685179,25700,0.128441
BMAA_rep2,32835279,44604873,0.736137,25672,0.107921
BMAA_rep3,23773774,35654113,0.666789,25449,0.120401
BMAA_rep4,25057094,34596848,0.724259,24966,0.102328
BMAA_rep5,31419960,44520337,0.705744,25411,0.117438


In [9]:
for sample in qc_df.index:
  # Absolute thresholds
  a_reasons = []
  if qc_df.loc[sample, 'total_reads'] < reads_threshold: a_reasons.append('low_reads')
  if qc_df.loc[sample, 'mapping_rate'] < mapping_threshold: a_reasons.append('low_mapping_rate')
  if qc_df.loc[sample, 'multimapping_rate'] > rRNA_threshold: a_reasons.append('high_multimapping_rate')
  if a_reasons ==[]: 
    qc_df.loc[sample, 'absolute_thresholds'] = 'Pass'
  else: 
    qc_df.loc[sample, 'absolute_thresholds'] = ','.join(a_reasons)
  # Relative thresholds 
  r_reasons = []
  if qc_df.loc[sample, 'total_reads'] < thresholds_df.loc['total_reads', 'mad_lower']: r_reasons.append('low_reads')
  if qc_df.loc[sample, 'mapping_rate'] < thresholds_df.loc['mapping_rate', 'mad_lower']: r_reasons.append('low_mapping_rate')
  if qc_df.loc[sample, 'multimapping_rate'] > thresholds_df.loc['multimapping_rate', 'mad_upper']: r_reasons.append('high_multimapping_rate')
  if qc_df.loc[sample, 'n_detected_genes'] < thresholds_df.loc['n_detected_genes', 'mad_lower']: r_reasons.append('low_n_detected_genes')
  if r_reasons ==[]:
    qc_df.loc[sample, 'relative_thresholds'] = 'Pass'
  else:
    qc_df.loc[sample, 'relative_thresholds'] = ','.join(r_reasons)
  
qc_df.to_csv(qc_out_path)